# Module 5: Decomposition into Trend, Season and Remainder

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Beginner [Topic 7](../Beginner/Topic_07_Trend.md) and [Topic 8](../Beginner/Topic_08_Seasonality.md)
argued that a series contains a trend and a season at the same time.
Decomposition does that separation arithmetically, and hands you the three
pieces as separate series you can chart, measure and model.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

## 2. Additive or multiplicative

Two ways the pieces can combine.

**Additive:** observed = trend + season + remainder. July adds 40 incidents.

**Multiplicative:** observed = trend × season × remainder. July is 40 percent
above trend.

For counts, multiplicative is almost always right. A department running at 100
a month and one running at 10 do not both get 40 extra incidents in July; they
both run about 40 percent high. Multiplying also keeps the pieces from
predicting negative counts.

The convenient trick: taking logs turns multiplication into addition, so a
multiplicative decomposition is an additive one on the log scale.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

add = seasonal_decompose(grandview, model="additive", period=12)
mul = seasonal_decompose(grandview, model="multiplicative", period=12)

print("additive: how many incidents July adds     ",
      round(add.seasonal.groupby(add.seasonal.index.month).mean().loc[7], 1))
print("multiplicative: how much July multiplies by",
      round(mul.seasonal.groupby(mul.seasonal.index.month).mean().loc[7], 2))

## 3. STL

`seasonal_decompose` uses a fixed moving average and assumes the seasonal shape
never changes. STL, which stands for seasonal and trend decomposition using
loess, is the better default: it lets the seasonal shape drift slowly over the
years, and with `robust=True` it refuses to let a single extraordinary month
distort everything around it.

Run it on the log of the series, which makes it multiplicative.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(np.log(grandview), period=12, robust=True).fit()

trend = np.exp(stl.trend)
season = np.exp(stl.seasonal)
remainder = np.exp(stl.resid)

print(f"trend at the start {trend.iloc[0]:6.1f}   at the end {trend.iloc[-1]:6.1f}")
print(f"seasonal factor range {season.min():.2f} to {season.max():.2f}")
within10 = 100 * ((remainder - 1).abs() < 0.10).mean()
print(f"remainder: {within10:.0f} percent of months land within 10 percent "
      f"of what the trend and season predicted")

## 4. The three pieces multiply back to the original

This is worth checking once, because it is the whole claim decomposition makes.

In [ ]:
rebuilt = trend * season * remainder
print(f"largest disagreement between the rebuilt series and the original: "
      f"{float((rebuilt - grandview).abs().max()):.6f}")

## 5. Reading the pieces

**The trend** is the level once the calendar and the noise are gone. Ashfell
falls from about 119 incidents a month to about 90.

**The season** is the repeating shape, as a multiplier. Below 1.0 means a quiet
month.

**The remainder** is everything the first two do not explain. It should look
like noise around 1.0. Structure left in the remainder means the decomposition
missed something.

In [ ]:
index = season.groupby(season.index.month).mean().round(2)
index.index = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print("average seasonal factor by month")
print(index.to_string())
print(f"\npeak month: {index.idxmax()} at {index.max():.2f}")

## 6. Which series you decompose changes the answer

Ashfell's **counts** peak in August. Its **rate per 100 arrests** peaks in
July. Both are correct, and the reason is that a count carries two seasonal
patterns at once: the rate has one, and the denominator has another.

In [ ]:
def seasonal_index(s):
    f = STL(np.log(s), period=12, robust=True).fit()
    return np.exp(f.seasonal).groupby(s.index.month).mean()

table = pd.DataFrame({
    "counts": seasonal_index(grandview),
    "arrests": seasonal_index(series("A012", "n_arrests")),
    "rate per 100 arrests": seasonal_index(rate("A012")),
}).round(2)
table.index = index.index
print(table.to_string())
print("\npeak month:", {c: table[c].idxmax() for c in table.columns})

Arrests peak in August and the rate peaks in July, so the count, which is the
two multiplied together, peaks in August. If the question is about **officer
behaviour**, decompose the rate. If it is about **workload**, decompose the
count. [Module 3](Module_03_Choosing_A_Denominator.md) is the same decision in
a different guise.

## 7. Checking against the answer key

This dataset was built with a known seasonal pattern in the use of force rate:
a peak in July, about 20 percent above the annual average. See
[Data/GROUND_TRUTH.md](../../../Data/GROUND_TRUTH.md).

In [ ]:
r = seasonal_index(rate("A012"))
print(f"recovered peak month     : {r.idxmax()}   built in: 7")
print(f"recovered amplitude      : {(r.max() - r.min()) / 2:.2f}   built in: 0.20")

The peak lands on the right month and the amplitude is in the right
neighbourhood. It is not exact, because one agency's seven years is a limited
amount of evidence about a seasonal shape, and because STL lets the shape drift.

## 8. What decomposition needs from you

| Requirement | Why | What happens otherwise |
|---|---|---|
| A complete series, no gaps | loess cannot span a hole | an error, or silently wrong output |
| At least two full cycles | the seasonal shape is estimated across years | unstable, meaningless factors |
| `robust=True` when outliers exist | one month otherwise bends the trend | a dip or bump around the event |
| Provisional months removed | the last points drag the trend down | a downturn that is not there |

In [ ]:
plot = pd.DataFrame({"observed": grandview, "trend": trend,
                     "season": season, "remainder": remainder})
axes = plot.plot(subplots=True, figsize=(9, 6.2), legend=False,
                 color=["#2a78d6", "#eb6834", "#1baf7a", "#8a8880"])
for ax, name in zip(axes, plot.columns):
    ax.set_ylabel(name, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].figure.tight_layout()

## Exercise

Tarnbridge has one extraordinary month, June 2021, when a week of civil unrest
produced 176 incidents against a typical 31. Decompose it twice, once with
`robust=True` and once without, and compare the trend around 2021.

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A002"

if AGENCY:
    s = series(AGENCY)
    soft = np.exp(STL(np.log(s), period=12, robust=False).fit().trend)
    hard = np.exp(STL(np.log(s), period=12, robust=True).fit().trend)
    out = pd.DataFrame({"observed": s, "trend, not robust": soft.round(1),
                        "trend, robust": hard.round(1)})
    print(out.loc["2021-01":"2021-12"].to_string())
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
```

Without `robust=True` the trend rises visibly through 2021, peaking near the
unrest month, and then falls again. The decomposition has absorbed a one week
event into the estimate of the agency's underlying level, which then contaminates
everything downstream: the seasonal factors, the remainder, and any trend
measured from it.

With `robust=True` the trend barely moves and the entire event lands in the
remainder, which is where a one off event belongs.

The general rule: find the outliers first, as in Beginner
[Topic 11](../Beginner/Topic_11_Outliers_And_Spikes.md), then choose a method
that will not be moved by them.

</details>

---

**Next:** [Module 6, Measuring the Trend](Module_06_Measuring_The_Trend.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*